# Week 38: Language Modelling

In [ ]:
import nb_setup

from src.data import load_tokenised_data
from src.lm_utils import build_vocab, apply_unk, UnigramLM, BigramLM, TrigramLM

import pandas as pd
import math

LANG = "ar"

df_train_tokenised, df_val_tokenised = load_tokenised_data()

train_df = df_train_tokenised.loc[df_train_tokenised["lang"] == LANG]
val_df   = df_val_tokenised.loc[df_val_tokenised["lang"] == LANG]

ar_train = train_df["question_tokens"].tolist()
ar_val   = val_df["question_tokens"].tolist()

In [ ]:
vocab = build_vocab(ar_train, min_count=2)

ar_train_unk = [apply_unk(toks, vocab) for toks in ar_train]
ar_val_unk   = [apply_unk(toks, vocab) for toks in ar_val]

print("Vocab size (incl. <UNK>/<s>/</s>):", len(vocab))

In [ ]:
def unk_rate(token_lists):
    total = sum(len(t) for t in token_lists)
    unks = sum(t.count("<UNK>") for t in token_lists)
    return unks / total

print("Val UNK rate:", unk_rate(ar_val_unk))

**Unigram Modelling with Add-k Smoothing**

In [ ]:
for k in [0.001, 0.005, 0.01, 0.02, 0.05]:
    m = UnigramLM(k=k)
    m.fit(ar_train_unk, vocab)
    print(f"k={k}: val perplexity = {m.perplexity(ar_val_unk):.2f}")

In [ ]:
BEST_K = 0.001 # lowest value
model = UnigramLM(k=BEST_K)
model.fit(ar_train_unk, vocab)
ppl = model.perplexity(ar_val_unk)
print(f"Unigram (k={BEST_K}) validation perplexity:", ppl)

In [ ]:
def question_perplexity(model, tokens):
    logprob = model.sentence_logprob(tokens)
    return math.exp(-logprob / len(tokens))

scored = sorted(
    zip(ar_val, ar_val_unk),
    key=lambda pair: question_perplexity(model, pair[1])
)
print("Lowest perplexity (most predictable):", scored[:3])
print("Highest perplexity (most surprising):", scored[-3:])

In [ ]:
def question_unk_rate(unk_tokens):
    return unk_tokens.count("<UNK>") / len(unk_tokens)

rows = []
for orig, unk in zip(ar_val, ar_val_unk):
    rows.append({
        "unk_rate": question_unk_rate(unk),
        "perplexity": question_perplexity(model, unk),
        "length": len(unk),
    })

diag_df = pd.DataFrame(rows)
print(diag_df[["unk_rate", "perplexity"]].corr())

In [ ]:
# perplexity on full validation set
full_ppl = model.perplexity(ar_val_unk)

# perplexity on de-duplicated validation set
dedup_mask = ~val_df["question"].duplicated()
ar_val_dedup_unk = [unk for unk, keep in zip(ar_val_unk, dedup_mask) if keep]
dedup_ppl = model.perplexity(ar_val_dedup_unk)

print("Full val perplexity:", full_ppl)
print("De-duplicated val perplexity:", dedup_ppl)

In [ ]:
results_row_amna = {
    "member": "amna",
    "language": LANG,
    "model": "unigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}
results_row_amna

# Results of Unigram with Add-K Smoothing

We implement a unigram baseline with add-k smoothing over Arabic questions, using a shared vocabulary built from training tokens (min-count 2, folding rarer tokens into <UNK>; vocab size 1,723). Sweeping k ∈ {0.001, 0.005, 0.01, 0.02, 0.05} shows validation perplexity increasing monotonically with k (80.81 at k=0.001 to 86.12 at k=2.0), with diminishing returns below k=0.01; we select k=0.001. We attribute this to the vocabulary/UNK strategy already absorbing most out-of-vocabulary risk (28.0% of validation tokens map to <UNK>), leaving little for additional smoothing to protect against, so heavier smoothing chiefly reallocates probability mass away from well-attested words. Per-question analysis shows a strong negative correlation (r = −0.68) between a question's <UNK> rate and its perplexity: questions dominated by <UNK> register as "predictable" because <UNK> itself is a high-frequency token in the model, not because the underlying question is linguistically simple a limitation of unigram modelling worth noting, since it rewards lexical genericness over topical specificity. The validation set contains 74/415 (17.8%) duplicate questions; de-duplicating changes perplexity only modestly (80.81 → 79.48), indicating the headline number is not substantially inflated by repeated examples.

### Bigram model with Add-K Smoothing

In [ ]:
bi_gram_ks = [0.001, 0.005, 0.01, 0.02, 0.05]
ppl_per_k = {}
for k in bi_gram_ks:
    model = BigramLM(k=k)

    model.fit(ar_train_unk, vocab)

    ppl = model.perplexity(ar_val_unk)
    ppl_per_k[k] = ppl

    print(
        f"k={k}: "
        f"validation perplexity = {ppl:.2f}"
    )

In [ ]:
BEST_K = min(ppl_per_k, key=ppl_per_k.get)

model = BigramLM(k=BEST_K)

model.fit(ar_train_unk, vocab)

full_ppl = model.perplexity(ar_val_unk)

In [ ]:
dedup_mask = ~val_df["question"].duplicated()

ar_val_dedup_unk = [
    unk
    for unk, keep in zip(ar_val_unk, dedup_mask)
    if keep
]

dedup_ppl = model.perplexity(
    ar_val_dedup_unk
)

print("Bigram Full validation perplexity:", full_ppl)
print("De-duplicated validation perplexity:", dedup_ppl)

In [ ]:
scored = [
    (
        original,
        unk,
        question_perplexity(model, unk),
    )
    for original, unk
    in zip(ar_val, ar_val_unk)
]

scored = sorted(scored, key=lambda x: x[2])

In [ ]:
for original, unk, ppl in scored[:5]:
    print(f"PPL={ppl:.2f}", original)

In [ ]:
for original, unk, ppl in scored[-5:]:
    print(f"PPL={ppl:.2f}", original)

In [ ]:
results_row_willy = {
    "member": "willy",
    "language": LANG,
    "model": "bigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}

results_row_willy

### Trigram Model with Add-K Smoothing

In [ ]:
trigram_ks = [0.001, 0.005, 0.01, 0.02, 0.05]

ppl_per_k = {}

for k in trigram_ks:
    model = TrigramLM(k=k)
    model.fit(ar_train_unk, vocab)
    ppl = model.perplexity(ar_val_unk)

    ppl_per_k[k] = ppl
    print(f"k={k}: validation perplexity = {ppl:.2f}")

NameError: name 'TrigramLM' is not defined

In [ ]:
BEST_K = min(ppl_per_k, key=ppl_per_k.get)
print("Best k:", BEST_K)

In [ ]:
model = TrigramLM(k=BEST_K)
model.fit(ar_train_unk, vocab)

In [ ]:
full_ppl = model.perplexity(ar_val_unk)

In [ ]:
dedup_mask = ~val_df["question"].duplicated()

ar_val_dedup_unk = [
    unk
    for unk, keep in zip(ar_val_unk, dedup_mask)
    if keep
]

dedup_ppl = model.perplexity(ar_val_dedup_unk)

print("Trigram Full validation perplexity:", full_ppl)
print("De-duplicated validation perplexity:", dedup_ppl)

In [ ]:
scored = [
    (
        original,
        unk,
        question_perplexity(model, unk),
    )
    for original, unk
    in zip(ar_val, ar_val_unk)
]

scored = sorted(scored, key=lambda x: x[2])

In [ ]:
# 5 most predictable questions
for original, unk, ppl in scored[:5]:
    print(f"PPL={ppl:.2f}", original)

In [ ]:
# 5 most surprising questions
for original, unk, ppl in scored[-5:]:
    print(f"PPL={ppl:.2f}", original)

In [ ]:
results_row_zarin = {
    "member": "zarin",
    "language": LANG,
    "model": "trigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}

results_row_zarin

In [ ]:
results_df = pd.DataFrame([
    results_row_amna,
    results_row_willy,
    results_row_zarin
])
results_df[
    ["val_unk_rate", "val_perplexity_full", "val_perplexity_dedup"]
] = results_df[
    ["val_unk_rate", "val_perplexity_full", "val_perplexity_dedup"]
].round(4)

results_df

In [ ]:
# checking